# Pandas: Time Series, Plotting, and Method Chaining

This notebook covers:
1. Time series basics, with dates in the index
2. Resampling (going from daily to weekly or monthly, for example)
3. Rolling windows and moving averages
4. Plotting directly from pandas
5. Method chaining for readable data pipelines
6. Performance considerations
7. A worked end-to-end example



In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Don't open a window — just save
import matplotlib.pyplot as plt
np.random.seed(42)
print("pandas:", pd.__version__)

pandas: 2.2.2


## Time Series - Dates as the Index

### definition

> "pandas contains extensive capabilities and features for working with time series data for all domains. Using the NumPy datetime64 and timedelta64 dtypes, pandas has consolidated a large number of features from other Python libraries..." - pandas docs

A few terms before the examples:
- **Time series**: a sequence of values measured over time, usually one value per time step (per day, per hour, etc.).
- **DatetimeIndex**: an Index made up of date or datetime values. When a DataFrame or Series has a DatetimeIndex, pandas knows how to slice it by year, month, or date range.
- **Frequency**: the spacing between time steps (daily, weekly, monthly, hourly, and so on).

The standard pattern for time series work in pandas is to put the dates in the index and then use the date-aware methods on the rest of the data.

In [2]:
# Create a daily time series
dates = pd.date_range('2024-01-01', periods=365, freq='D')
values = 100 + np.cumsum(np.random.randn(365) * 2)

ts = pd.Series(values, index=dates, name='price')
print(ts.head())
print(f"\nIndex type: {type(ts.index).__name__}")

2024-01-01    100.993428
2024-01-02    100.716900
2024-01-03    102.012277
2024-01-04    105.058336
2024-01-05    104.590030
Freq: D, Name: price, dtype: float64

Index type: DatetimeIndex


### Slicing time series - pandas understands dates

With a DatetimeIndex, rows can be selected using date strings. A full date selects a single row, a partial date (year-month or year) selects all rows in that range.

In [3]:
# Select a date range using strings
print("First week of January 2024:")
print(ts['2024-01-01':'2024-01-07'])

First week of January 2024:
2024-01-01    100.993428
2024-01-02    100.716900
2024-01-03    102.012277
2024-01-04    105.058336
2024-01-05    104.590030
2024-01-06    104.121756
2024-01-07    107.280181
Freq: D, Name: price, dtype: float64


In [4]:
# Even partial dates work
print("All of March 2024 (first 5 days):")
print(ts['2024-03'].head())

print("\nAll of 2024 — count:")
print(len(ts['2024']))

All of March 2024 (first 5 days):
2024-03-01    80.483090
2024-03-02    80.111772
2024-03-03    77.899102
2024-03-04    75.506688
2024-03-05    77.131740
Freq: D, Name: price, dtype: float64

All of 2024 — count:
365


## Part 2: Resampling - Changing Frequency

### definition

> "Resample time-series data. Convenience method for frequency conversion and resampling of time series. The object must have a datetime-like index... or pass datetime-like values to the on or level keyword." - pandas docs

**Resampling** is converting a time series from one frequency to another, like going from daily data to weekly or monthly. It is the time-series equivalent of `groupby`: each new period becomes a group, and an aggregation reduces the values inside it to one number.

In [5]:
# Convert daily to weekly (mean of each week)
weekly = ts.resample('W').mean()
print("Daily → weekly (mean):")
print(weekly.head())
print(f"\nDaily count: {len(ts)}, Weekly count: {len(weekly)}")

Daily → weekly (mean):
2024-01-07    103.538987
2024-01-14    107.448119
2024-01-21     97.092192
2024-01-28     92.630767
2024-02-04     89.670873
Freq: W-SUN, Name: price, dtype: float64

Daily count: 365, Weekly count: 53


In [6]:
# Monthly OHLC (open, high, low, close)
monthly = ts.resample('ME').agg(['first', 'max', 'min', 'last'])
monthly.columns = ['open', 'high', 'low', 'close']
print(monthly.round(2))

              open    high    low   close
2024-01-31  100.99  108.96  87.51   87.51
2024-02-29   91.21   91.21  75.98   81.44
2024-03-31   80.48   88.00  75.51   82.41
2024-04-30   84.35   84.35  72.95   82.58
2024-05-31   80.76   86.32  73.66   76.50
2024-06-30   75.14   93.28  74.76   91.57
2024-07-31   89.42  101.29  83.69  101.29
2024-08-31  102.59  105.28  96.59   99.77
2024-09-30   97.37  104.41  89.98   92.60
2024-10-31   90.64   96.67  90.13   96.57
2024-11-30   96.80  113.58  96.80  112.51
2024-12-31  112.22  112.22  99.89  107.26


**Common frequency codes:**

| Code | Meaning |
|---|---|
| `D` | Daily |
| `W` | Weekly |
| `ME` | Month end |
| `QE` | Quarter end |
| `YE` | Year end |
| `h` | Hourly |
| `min` | Minute |

## Rolling Windows and Moving Averages

### definition

> "Provide rolling window calculations. A rolling window is a fixed-size sliding window over a sequence. Rolling lets you compute aggregations over a moving subset of the data." - pandas docs

A few terms before the examples:
- **Window**: a fixed-size slice of consecutive rows.
- **Rolling window**: a window that slides forward one step at a time across the series.
- **Moving average**: the most common rolling statistic; the mean of the values inside the window.

Rolling windows are used to smooth noisy data, compute lagged features for machine learning, and detect trends in time series.

In [7]:
# 7-day moving average
ma7 = ts.rolling(window=7).mean()
ma30 = ts.rolling(window=30).mean()

# Combine for comparison
comparison = pd.DataFrame({
    'price': ts,
    'MA_7':  ma7,
    'MA_30': ma30
})
print(comparison.head(35).round(2))

             price    MA_7  MA_30
2024-01-01  100.99     NaN    NaN
2024-01-02  100.72     NaN    NaN
2024-01-03  102.01     NaN    NaN
2024-01-04  105.06     NaN    NaN
2024-01-05  104.59     NaN    NaN
2024-01-06  104.12     NaN    NaN
2024-01-07  107.28  103.54    NaN
2024-01-08  108.82  104.66    NaN
2024-01-09  107.88  105.68    NaN
2024-01-10  108.96  106.67    NaN
2024-01-11  108.03  107.10    NaN
2024-01-12  107.10  107.46    NaN
2024-01-13  107.59  107.95    NaN
2024-01-14  103.76  107.45    NaN
2024-01-15  100.31  106.23    NaN
2024-01-16   99.19  104.99    NaN
2024-01-17   97.16  103.31    NaN
2024-01-18   97.79  101.84    NaN
2024-01-19   95.97  100.25    NaN
2024-01-20   93.15   98.19    NaN
2024-01-21   96.08   97.09    NaN
2024-01-22   95.63   96.42    NaN
2024-01-23   95.76   95.93    NaN
2024-01-24   92.91   95.33    NaN
2024-01-25   91.82   94.48    NaN
2024-01-26   92.05   93.91    NaN
2024-01-27   89.74   93.43    NaN
2024-01-28   90.50   92.63    NaN
2024-01-29   8

Moving averages smooth out short-term noise so the underlying trend becomes easier to see. The first values are NaN because the window does not yet have enough data; once enough rows have passed, every value is the mean of the last N rows.

The same `.rolling()` method works with any aggregation, not just `mean` (`std`, `sum`, `max`, custom functions, and so on).

In [8]:
# Rolling can use any aggregation
print("7-day rolling stats (last 10 days):")
print(pd.DataFrame({
    'mean': ts.rolling(7).mean(),
    'std':  ts.rolling(7).std(),
    'max':  ts.rolling(7).max(),
    'min':  ts.rolling(7).min(),
}).tail(10).round(2))

7-day rolling stats (last 10 days):
              mean   std     max     min
2024-12-21  103.08  2.14  105.18   99.89
2024-12-22  103.49  1.65  105.18  100.51
2024-12-23  103.73  1.20  105.18  102.20
2024-12-24  103.64  1.25  105.18  102.20
2024-12-25  103.07  1.34  104.86  101.19
2024-12-26  102.70  1.10  104.82  101.19
2024-12-27  102.76  1.26  105.29  101.19
2024-12-28  103.09  1.53  105.29  101.19
2024-12-29  103.53  1.85  105.88  101.19
2024-12-30  104.25  2.20  107.26  101.19


In [9]:
# Expanding windows — growing from the start
print("Cumulative max so far:")
print(ts.expanding().max().tail(10).round(2))

Cumulative max so far:
2024-12-21    113.58
2024-12-22    113.58
2024-12-23    113.58
2024-12-24    113.58
2024-12-25    113.58
2024-12-26    113.58
2024-12-27    113.58
2024-12-28    113.58
2024-12-29    113.58
2024-12-30    113.58
Freq: D, Name: price, dtype: float64


## Plotting Directly From pandas

pandas has a `.plot()` method built into Series and DataFrame. It calls Matplotlib under the hood, so any Matplotlib knowledge carries over, and the resulting figure can be customised with Matplotlib commands.

For quick exploration, `.plot()` is usually all that is needed. Matplotlib directly is the right tool for detailed customisation, multi-panel figures, or publication-quality output.

In [17]:
# A single line plot
fig, ax = plt.subplots(figsize=(10, 4))
ts.plot(ax=ax, title='Daily Price Over 2024')
ax.set_ylabel('Price')
plt.tight_layout()
plt.savefig('plot1.png', dpi=80)
plt.close()
print("Plot saved to plot1.png — line plot of daily prices")

Plot saved to plot1.png — line plot of daily prices


In [14]:
# Multiple lines on the same plot
fig, ax = plt.subplots(figsize=(10, 4))
comparison[['price', 'MA_7', 'MA_30']].plot(ax=ax, title='Price with Moving Averages')
ax.set_ylabel('Price')
plt.tight_layout()
plt.savefig('plot2.png', dpi=80)
plt.close()
print("Saved plot with price + 2 moving averages")

Saved plot with price + 2 moving averages


In [15]:
# Bar chart from a groupby result
np.random.seed(0)
demo = pd.DataFrame({
    'region':  np.random.choice(['N', 'S', 'E', 'W'], 100),
    'revenue': np.random.randint(100, 1000, 100)
})

fig, ax = plt.subplots(figsize=(7, 4))
demo.groupby('region')['revenue'].sum().plot(kind='bar', ax=ax, title='Revenue by Region')
ax.set_ylabel('Total Revenue')
plt.tight_layout()
plt.savefig('plot3.png', dpi=80)
plt.close()
print("Saved bar chart of revenue by region")

Saved bar chart of revenue by region


In [18]:
# Histogram
fig, ax = plt.subplots(figsize=(7, 4))
ts.plot(kind='hist', bins=30, ax=ax, title='Distribution of Daily Prices', edgecolor='black')
ax.set_xlabel('Price')
plt.tight_layout()
plt.savefig('plot4.png', dpi=80)
plt.close()
print("Saved histogram")

Saved histogram


**pandas plot types** (passed via `kind=...`):

| kind | Use case |
|---|---|
| `line` (default) | Time series, trends |
| `bar` / `barh` | Categorical comparisons |
| `hist` | Distribution of one variable |
| `box` | Distribution comparison across groups |
| `scatter` | Relationship between two variables |
| `kde` | Smooth density estimate |
| `area` | Stacked time series |
| `pie` | Composition (rarely the clearest choice) |

Matplotlib will be covered in detail in a later notebook series. For day-to-day exploration, pandas plotting is enough.

## Method Chaining

### definition

> "Method chaining is a programming style where multiple method calls are made on the same object, one after the other, with each method returning the object so that the next call can operate on it." - common Python usage

In pandas, most DataFrame methods return a new DataFrame, which means they can be chained together: every step transforms the data and passes the result to the next step. Compared to creating intermediate variables (`df1`, `df2`, `df3`, ...) the chained version reads as a single top-to-bottom sequence of operations.

In [19]:
# Setup — sample order data
np.random.seed(0)
n = 500
orders = pd.DataFrame({
    'date':     pd.date_range('2024-01-01', periods=n, freq='6h'),
    'region':   np.random.choice(['North', 'South', 'East', 'West'], n),
    'product':  np.random.choice(['A', 'B', 'C', 'D'], n),
    'qty':      np.random.randint(1, 20, n),
    'price':    np.random.choice([100, 200, 500], n),
})
orders.loc[orders.sample(20).index, 'price'] = np.nan   # add some missing

print(orders.head())

                 date region product  qty  price
0 2024-01-01 00:00:00  North       D    8  100.0
1 2024-01-01 06:00:00   West       D   11  500.0
2 2024-01-01 12:00:00  South       A   19  200.0
3 2024-01-01 18:00:00  North       A    4  500.0
4 2024-01-02 00:00:00   West       C   13  100.0


In [20]:
# The verbose way — lots of intermediate variables
orders_clean = orders.copy()
orders_clean['price'] = orders_clean['price'].fillna(orders_clean['price'].median())
orders_clean['revenue'] = orders_clean['qty'] * orders_clean['price']
orders_recent = orders_clean[orders_clean['date'] >= '2024-02-01']
by_region = orders_recent.groupby('region')['revenue'].sum()
top_regions = by_region.sort_values(ascending=False)

print("Verbose approach result:")
print(top_regions)

Verbose approach result:
region
West     287400.0
South    282300.0
East     234100.0
North    205400.0
Name: revenue, dtype: float64


In [21]:
# Method chained — same result, much cleaner
top_regions = (
    orders
    .assign(
        price=lambda d: d['price'].fillna(d['price'].median()),
        revenue=lambda d: d['qty'] * d['price'],
    )
    .query("date >= '2024-02-01'")
    .groupby('region')['revenue']
    .sum()
    .sort_values(ascending=False)
)

print("Chained approach result:")
print(top_regions)

Chained approach result:
region
West     287400.0
South    282300.0
East     234100.0
North    205400.0
Name: revenue, dtype: float64


**Tips for method chaining:**
- Wrap the whole chain in `( ... )` so it can be broken across multiple lines.
- Use `.assign()` to create new columns inline.
- Inside `.assign()`, use `lambda d: ...` to refer to the DataFrame as it stands at that point in the chain (after previous steps have run).
- Keep each line doing one logical operation.
- For debugging, temporarily break the chain into pieces; once the logic works, glue it back together.

## Performance

### definition

> "Vectorization is the process of converting an algorithm from operating on a single value at a time to operating on a set of values at one time." - general programming usage

pandas is fast when operations are **vectorized** (applied to whole columns at once, using the NumPy machinery underneath). It is slow when operations go row by row in pure Python. The next cell demonstrates the gap with a small benchmark.

In [22]:
import time

n = 100_000
df = pd.DataFrame({
    'a': np.random.randint(0, 100, n),
    'b': np.random.randint(0, 100, n),
})

# Slow: iterating with a Python loop
start = time.time()
result_slow = []
for i, row in df.iterrows():
    result_slow.append(row['a'] + row['b'])
t_slow = time.time() - start

# Fast: vectorized
start = time.time()
result_fast = df['a'] + df['b']
t_fast = time.time() - start

print(f"iterrows:    {t_slow:.4f}s")
print(f"Vectorized:  {t_fast:.4f}s")
print(f"Speedup:     {t_slow/t_fast:.0f}x")

iterrows:    14.7425s
Vectorized:  0.0009s
Speedup:     16480x


**Performance rules of thumb:**

| Approach | Speed | Notes |
|---|---|---|
| **Vectorized operations** like `df['a'] + df['b']` | Fastest | NumPy under the hood. The default choice. |
| **Built-in methods** like `df.sum()`, `df['col'].str.upper()` | Fast | Implemented in C, optimised internally. |
| **`.apply()` with a Python function** | Slower | Useful when no vectorized equivalent exists. |
| **`.iterrows()` or for-loops** | Slowest | Pure Python row-by-row; avoid for large data. |

A useful rule: if a snippet starts with `for row in df.iterrows():`, there is almost always a faster way using vectorized operations, `.apply()`, or a built-in method.

### Other performance ideas

- Use `dtype='category'` for columns that repeat the same string values many times.
- For very large CSVs, read in chunks with `chunksize=...`.
- For data that does not fit in memory, look at Polars or Dask.
- For complex filters on large DataFrames, `query()` can be faster than building a boolean mask.

## A Worked End-to-End Analysis

This section walks through a realistic small analysis from start to finish:
1. Generate some messy data (missing values, mixed capitalisation)
2. Clean it
3. Derive useful columns
4. Aggregate to monthly insights by customer type
5. Visualise the result

The whole pipeline is written as a single chained expression where possible.

In [23]:
# Generate fake e-commerce data
np.random.seed(42)
n = 2000

raw = pd.DataFrame({
    'order_id':   range(1, n + 1),
    'date':       pd.date_range('2024-01-01', periods=n, freq='3h'),
    'customer':   np.random.choice(['New', 'Returning', 'VIP'], n, p=[0.5, 0.4, 0.1]),
    'category':   np.random.choice(['Electronics', 'Clothing', 'Books', 'Home'], n),
    'qty':        np.random.randint(1, 5, n),
    'unit_price': np.round(np.random.uniform(10, 200, n), 2),
})

# Inject some realistic messiness
raw.loc[raw.sample(50, random_state=1).index, 'unit_price'] = np.nan
raw.loc[raw.sample(30, random_state=2).index, 'qty'] = np.nan
raw['category'] = raw['category'].sample(frac=1, random_state=0).reset_index(drop=True)
# Random capitalization
mask = raw.sample(frac=0.3, random_state=3).index
raw.loc[mask, 'category'] = raw.loc[mask, 'category'].str.upper()

print("Raw data sample:")
print(raw.head())
print(f"\nShape: {raw.shape}")
print(f"\nMissing:\n{raw.isna().sum()}")

Raw data sample:
   order_id                date   customer     category  qty  unit_price
0         1 2024-01-01 00:00:00        New  Electronics  3.0      118.68
1         2 2024-01-01 03:00:00        VIP        BOOKS  1.0      163.03
2         3 2024-01-01 06:00:00  Returning         Home  1.0      154.43
3         4 2024-01-01 09:00:00  Returning  ELECTRONICS  4.0       39.24
4         5 2024-01-01 12:00:00        New         HOME  3.0       38.36

Shape: (2000, 6)

Missing:
order_id       0
date           0
customer       0
category       0
qty           30
unit_price    50
dtype: int64


In [24]:
# Full analysis pipeline
insights = (
    raw
    # 1. Clean
    .assign(
        qty=lambda d: d['qty'].fillna(1).astype(int),
        unit_price=lambda d: d['unit_price'].fillna(d['unit_price'].median()),
        category=lambda d: d['category'].str.title(),
    )
    # 2. Derive
    .assign(
        revenue=lambda d: d['qty'] * d['unit_price'],
        month=lambda d: d['date'].dt.to_period('M'),
    )
    # 3. Aggregate
    .groupby(['month', 'customer'])
    .agg(
        revenue=('revenue', 'sum'),
        orders=('order_id', 'count'),
        avg_order=('revenue', 'mean'),
    )
    .round(2)
)

print("Monthly insights by customer type:")
print(insights.head(15))

Monthly insights by customer type:
                    revenue  orders  avg_order
month   customer                              
2024-01 New        30086.55     124     242.63
        Returning  22789.83      99     230.20
        VIP         7011.21      25     280.45
2024-02 New        25067.42     108     232.11
        Returning  25589.42      98     261.12
        VIP         8152.92      26     313.57
2024-03 New        35211.75     129     272.96
        Returning  22335.57      93     240.17
        VIP         6545.48      26     251.75
2024-04 New        33014.05     125     264.11
        Returning  24381.22      95     256.64
        VIP         4276.32      20     213.82
2024-05 New        30828.98     112     275.26
        Returning  31178.55     107     291.39
        VIP         7787.85      29     268.55


In [25]:
# Pivot for a cleaner cross-tab view
pivot = (
    raw
    .assign(
        qty=lambda d: d['qty'].fillna(1).astype(int),
        unit_price=lambda d: d['unit_price'].fillna(d['unit_price'].median()),
        category=lambda d: d['category'].str.title(),
        revenue=lambda d: d['qty'] * d['unit_price'],
        month=lambda d: d['date'].dt.to_period('M').astype(str),
    )
    .pivot_table(
        index='month',
        columns='customer',
        values='revenue',
        aggfunc='sum',
    )
    .round(0)
)

print("Monthly revenue by customer type:")
print(pivot)

Monthly revenue by customer type:
customer      New  Returning     VIP
month                               
2024-01   30087.0    22790.0  7011.0
2024-02   25067.0    25589.0  8153.0
2024-03   35212.0    22336.0  6545.0
2024-04   33014.0    24381.0  4276.0
2024-05   30829.0    31179.0  7788.0
2024-06   30765.0    24048.0  4749.0
2024-07   29951.0    25472.0  5875.0
2024-08   34911.0    28258.0  5523.0
2024-09    8535.0     4789.0  1015.0


In [27]:
# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
pivot.plot(kind='bar', ax=ax, title='Monthly Revenue by Customer Type')
ax.set_ylabel('Revenue ($)')
ax.set_xlabel('Month')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('final_analysis.png', dpi=80)
plt.close()
print("Saved final analysis chart")

Saved final analysis chart


## Capstone Exercises

The exercises below use the weather dataset created in the next cell. Each one practises tools from across the five notebooks: groupby, joins, rolling windows, plotting, categoricals, and method chaining.

In [28]:
# Capstone setup — a realistic dataset
np.random.seed(0)
n_days = 365
dates = pd.date_range('2024-01-01', periods=n_days, freq='D')

weather = pd.DataFrame({
    'date': dates,
    'city': np.random.choice(['NYC', 'LA', 'Chicago', 'Houston'], n_days),
    'temp_celsius': np.round(20 + 10*np.sin(np.arange(n_days)*2*np.pi/365)
                             + np.random.normal(0, 3, n_days), 1),
    'humidity': np.random.randint(30, 90, n_days),
    'precipitation_mm': np.round(np.random.exponential(2, n_days), 1),
})

# Add some missing values
weather.loc[weather.sample(20, random_state=1).index, 'humidity'] = np.nan

print(weather.head())
print(f"\nShape: {weather.shape}")

        date     city  temp_celsius  humidity  precipitation_mm
0 2024-01-01      NYC          22.7      37.0               2.5
1 2024-01-02  Houston          16.7      85.0               0.3
2 2024-01-03       LA          21.5      56.0               0.8
3 2024-01-04      NYC          22.2      78.0               2.6
4 2024-01-05  Houston          20.9      37.0               2.2

Shape: (365, 5)


### Problem 1
Find the average temperature per city. Then find which city had the highest single-day temperature ever in this dataset.

### Problem 2
For each city, compute the monthly average temperature and humidity. Present as a tidy DataFrame.

### Problem 3
Compute a 7-day rolling average of temperature for NYC only. Plot it alongside the daily temperature.

### Problem 4
Categorize each day's precipitation as 'Dry' (< 1mm), 'Light' (1-5mm), 'Moderate' (5-15mm), or 'Heavy' (> 15mm). Then count how many days of each type each city had.

### Problem 5 (challenge)
Using method chaining, produce a single DataFrame showing each city's:
- Mean temperature
- Total precipitation
- Number of "Heavy" precipitation days
- Number of days where humidity exceeded 80%

In ONE chained expression.

In [ ]:
# Problem 1
print("Average temp per city:")
print(weather.groupby('city')['temp_celsius'].mean().round(2).sort_values(ascending=False))

# Highest single-day temp
hottest = weather.loc[weather['temp_celsius'].idxmax()]
print(f"\nHottest day: {hottest['date'].date()} in {hottest['city']}: {hottest['temp_celsius']}°C")

Average temp per city:
city
NYC        20.62
Chicago    20.18
Houston    20.13
LA         19.36
Name: temp_celsius, dtype: float64

Hottest day: 2024-03-11 in LA: 37.3°C


In [ ]:
# Problem 2
weather['month'] = weather['date'].dt.to_period('M')
result = (
    weather
    .groupby(['city', 'month'])
    .agg(avg_temp=('temp_celsius', 'mean'),
         avg_humidity=('humidity', 'mean'))
    .round(2)
)
print(result.head(15))

                 avg_temp  avg_humidity
city    month                          
Chicago 2024-01     24.80         55.00
        2024-02     28.98         54.75
        2024-03     28.86         69.57
        2024-04     27.16         64.86
        2024-05     24.70         57.67
        2024-06     23.25         58.55
        2024-07     18.24         61.11
        2024-08     12.08         46.33
        2024-09     12.26         59.20
        2024-10      9.90         64.50
        2024-11     12.64         43.25
        2024-12     18.67         75.40
Houston 2024-01     21.07         63.62
        2024-02     26.79         52.92
        2024-03     30.41         64.62


In [32]:
# Problem 3
nyc = weather[weather['city'] == 'NYC'].sort_values('date').reset_index(drop=True)
nyc['temp_ma7'] = nyc['temp_celsius'].rolling(7).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(nyc['date'], nyc['temp_celsius'], alpha=0.4, label='Daily')
ax.plot(nyc['date'], nyc['temp_ma7'], color='red', label='7-day MA')
ax.legend()
ax.set_title('NYC Temperature 2024')
ax.set_ylabel('°C')
plt.tight_layout()
plt.savefig('nyc_temp.png', dpi=80)
plt.close()
print("Saved NYC temperature chart")

Saved NYC temperature chart


In [33]:
# Problem 4
weather['precip_category'] = pd.cut(
    weather['precipitation_mm'],
    bins=[-0.01, 1, 5, 15, float('inf')],
    labels=['Dry', 'Light', 'Moderate', 'Heavy']
)

result = weather.groupby(['city', 'precip_category'], observed=True).size().unstack(fill_value=0)
print("Days of each precipitation type per city:")
print(result)

Days of each precipitation type per city:
precip_category  Dry  Light  Moderate
city                                 
Chicago           32     39        10
Houston           38     59         7
LA                34     42        11
NYC               44     45         4


In [ ]:
# Problem 5
summary = (
    weather
    .assign(
        is_heavy=lambda d: d['precipitation_mm'] > 15,
        high_humidity=lambda d: d['humidity'] > 80,
    )
    .groupby('city')
    .agg(
        mean_temp=('temp_celsius', 'mean'),
        total_precipitation=('precipitation_mm', 'sum'),
        heavy_days=('is_heavy', 'sum'),
        high_humidity_days=('high_humidity', 'sum'),
    )
    .round(2)
)
print(summary)

         mean_temp  total_precipitation  heavy_days  high_humidity_days
city                                                                   
Chicago      20.18                185.1           0                  16
Houston      20.13                209.4           0                  20
LA           19.36                196.1           0                  11
NYC          20.62                166.4           0                  12
